# EEG Seizure Correlation & Coherence Pipeline

Just runs things in order. The logic is in `config`, `obe_io`, `matching`, `correlation`, `coherence` and `plotting`.

## 1. Imports & Configuration

Modules, every `config` constant, and the save/display helpers the summaries use.

In [ ]:
%matplotlib inline
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta
from IPython.display import display

from config import *
from obe_io import read_data
from matching import load_seizure_metadata, match_obe_files
from correlation import (
    channel_indices,
    datetime_to_sample,
    compute_rolling_correlation,
    filter_by_baseline,
    offset_to_day_zero,
)
from coherence import compute_seizure_coherence
import plotting as P


def render_per_animal(results, out_dir, name_fn, build_fn):
    os.makedirs(out_dir, exist_ok=True)
    figs = []
    for animal, group in results.groupby('Animal_ID'):
        fig = build_fn(group)
        fig.savefig(os.path.join(out_dir, name_fn(animal)))
        display(fig)
        plt.close(fig)
        figs.append(fig)
    return figs


def render_global(out_dir, filename, fig):
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, filename))
    display(fig)
    plt.close(fig)

## 2. Load Seizure Metadata

Read the seizure sheet and parse start/end datetimes.

In [ ]:
seizure_df = load_seizure_metadata(CSV_PATH)
print(f"Loaded {len(seizure_df)} seizure records")
seizure_df.head()

## 3. Match OBE Files to Seizures

Pair each `.OBE` recording with the seizures that overlap it. Header times are 1 hour off (DST).

In [ ]:
matched_results = match_obe_files(FOLDER_PATH, seizure_df, DST_OFFSET_HOURS)

## 4. Matched Seizure Summary

Matched files with animal, recording order and seizure count.

In [ ]:
total = sum(len(m.seizures) for m in matched_results)
print(f"{len(matched_results)} files matched, {total} seizures total (1hr offset applied)\n")
for m in matched_results:
    s = m.seizures
    print(f"{m.filename}: Animal {s['Animal_no'].iloc[0]}, "
          f"Rec {s['Recording_order'].iloc[0]}, {len(s)} seizure(s)")

## 5. 60-Minute EEG Overview Plots

Both channels per file, decimated, with seizure markers.

In [ ]:
os.makedirs(OVERVIEW_OUTPUT_DIR, exist_ok=True)
overview_plots = []

for m in matched_results:
    row = m.seizures.iloc[0]
    ch1, ch2 = channel_indices(row['Recording_order'], CH1_OFFSET, CH2_OFFSET, CHANNEL_STRIDE)
    path = os.path.join(FOLDER_PATH, m.filename)
    d1, d2 = read_data(path, ch1), read_data(path, ch2)
    fig = P.plot_overview(d1, d2, m.seizures, m.corrected_start, m.header.samp_freq,
                          ch1, ch2, m.filename, row['Animal_no'], row['Recording_order'],
                          DOWNSAMPLE_FACTOR, NUM_MINS_PER_SUBPLOT, OVERVIEW_NUM_SUBPLOTS)
    safe = re.sub(r'[\\/:*?"<>|]', '_', m.filename)
    fig.savefig(os.path.join(OVERVIEW_OUTPUT_DIR,
                f"Animal{row['Animal_no']}_Rec{row['Recording_order']}_{safe}.png"))
    overview_plots.append(fig)
    plt.close(fig)

print(f"{len(overview_plots)} overview plots generated.")

In [ ]:
for fig in overview_plots:
    display(fig)

## 6. Baseline Correlation Pre-Filter

Drop seizures whose pre-seizure baseline is already too correlated, or whose window falls outside the recording.

In [ ]:
seizures_to_analyze, excluded_seizures = filter_by_baseline(
    matched_results, FOLDER_PATH,
    baseline_duration_secs=BASELINE_DURATION_SECS,
    baseline_corr_threshold=BASELINE_CORR_THRESHOLD,
    rolling_window_secs=ROLLING_WINDOW_SECS,
    rolling_step_secs=ROLLING_STEP_SECS,
    ch1_offset=CH1_OFFSET, ch2_offset=CH2_OFFSET, channel_stride=CHANNEL_STRIDE,
)

## 7. Main Analysis Loop

Per seizure: rolling correlation over a padded window, then during vs pre-seizure coherence and phase delay.

In [ ]:
os.makedirs(CORRELATION_SEIZURE_DIR, exist_ok=True)
os.makedirs(COHERENCE_SEIZURE_DIR, exist_ok=True)
os.makedirs(PHASE_DELAY_SEIZURE_DIR, exist_ok=True)

# decision: the pre-filter is optional, without it every seizure runs and baseline corr is NaN
if 'seizures_to_analyze' not in globals():
    seizures_to_analyze = [(mi, ri, float('nan'))
                           for mi, m in enumerate(matched_results)
                           for ri in range(len(m.seizures))]
    print(f"No baseline pre-filter run; analyzing all {len(seizures_to_analyze)} seizures.")

correlation_rows, coherence_rows = [], []
correlation_plots, coherence_plots, phase_delay_plots = [], [], []
coherence_details = []

for n, (match_idx, row_idx, baseline_corr) in enumerate(seizures_to_analyze, start=1):
    m = matched_results[match_idx]
    row = m.seizures.iloc[row_idx]
    path = os.path.join(FOLDER_PATH, m.filename)
    sf = m.header.samp_freq
    ch1, ch2 = channel_indices(row['Recording_order'], CH1_OFFSET, CH2_OFFSET, CHANNEL_STRIDE)
    print(f"[{n}] Seizure {row['Seizure']} (Animal {row['Animal_no']})")

    try:
        d1_full, d2_full = read_data(path, ch1), read_data(path, ch2)

        midpoint = row['SeizureStartDT'] + (row['SeizureEndDT'] - row['SeizureStartDT']) / 2
        pad_start = midpoint - timedelta(seconds=PADDING_WINDOW_SECS)
        pad_end = midpoint + timedelta(seconds=PADDING_WINDOW_SECS)
        start_samp = datetime_to_sample(pad_start, m.corrected_start, sf)
        end_samp = datetime_to_sample(pad_end, m.corrected_start, sf)
        if start_samp < 0 or end_samp > m.header.samples:
            print(f"  ! Seizure {row['Seizure']} window out of bounds. Skipping.")
            continue

        data1, data2 = d1_full[start_samp:end_samp], d2_full[start_samp:end_samp]
        corrs, time_axis = compute_rolling_correlation(
            data1, data2, sf, ROLLING_WINDOW_SECS, ROLLING_STEP_SECS)

        sz_start_sec = (row['SeizureStartDT'] - pad_start).total_seconds()
        sz_end_sec = (row['SeizureEndDT'] - pad_start).total_seconds()
        seiz_corrs = corrs[(time_axis >= sz_start_sec) & (time_axis <= sz_end_sec)]
        seconds_above = int(np.sum(np.abs(seiz_corrs) > CORR_THRESHOLD)) * ROLLING_STEP_SECS
        avg_corr = float(np.mean(np.abs(seiz_corrs))) if seiz_corrs.size else 0.0
        seiz_dur = (row['SeizureEndDT'] - row['SeizureStartDT']).total_seconds()

        correlation_rows.append(dict(
            OBE_File=m.filename, File_Start_DT=m.corrected_start, Animal_ID=row['Animal_no'],
            Racine_Scale=int(row['Racine_Scale']), Seconds_Above_Threshold=seconds_above,
            Seizure_Duration_Sec=float(seiz_dur), Avg_Corr_Strength=avg_corr, Ch1=ch1, Ch2=ch2))

        ss = datetime_to_sample(row['SeizureStartDT'], pad_start, sf)
        se = datetime_to_sample(row['SeizureEndDT'], pad_start, sf)
        raw1, raw2 = data1[ss:se], data2[ss:se]
        base_str = 'N/A' if np.isnan(baseline_corr) else round(baseline_corr, 2)
        title = f"Animal {row['Animal_no']} | Sec Above: {round(seconds_above, 1)}s | Base Corr: {base_str}"
        date_str = row['SeizureStartDT'].strftime('%Y-%m-%d')
        phase_plot_path = os.path.join(PHASE_DELAY_SEIZURE_DIR,
            f"PhaseDelay_Animal_{row['Animal_no']}_Seiz_{row['Seizure']}_{date_str}.png")
        coh_plot_path = os.path.join(COHERENCE_SEIZURE_DIR,
            f"Coherence_Animal_{row['Animal_no']}_Seiz_{row['Seizure']}_{date_str}.png")
        corr_plot_path = os.path.join(CORRELATION_SEIZURE_DIR,
            f"Animal_{row['Animal_no']}_Seiz_{row['Seizure']}_{date_str}.png")
        fig = P.assemble_correlation_figure(
            corrs, time_axis, sz_start_sec, sz_end_sec, raw1, raw2, sf,
            CORR_THRESHOLD, SEGMENT_SECS, title=title)
        fig.savefig(corr_plot_path)
        correlation_plots.append(fig)
        plt.close(fig)

        pre_start_s = datetime_to_sample(
            row['SeizureStartDT'] - timedelta(seconds=PRE_SEIZURE_SECS), m.corrected_start, sf)
        sz_start_s = datetime_to_sample(row['SeizureStartDT'], m.corrected_start, sf)
        sz_end_s = datetime_to_sample(row['SeizureEndDT'], m.corrected_start, sf)
        if pre_start_s >= 0 and sz_end_s <= m.header.samples:
            coh = compute_seizure_coherence(
                d1_full, d2_full, sz_start_s, sz_end_s, pre_start_s, sz_start_s, sf,
                COH_WINDOW, COH_NOVERLAP, COH_NFFT, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ,
                phase_msc_threshold=PHASE_MSC_THRESHOLD,)
            coherence_rows.append(dict(
                OBE_File=m.filename,
                Animal_ID=row['Animal_no'],
                Seizure=int(row['Seizure']),
                File_Start_DT=m.corrected_start,
                Racine_Scale=int(row['Racine_Scale']),
                Seizure_Duration_Sec=float(seiz_dur),
                Avg_MSC_Seizure_Low=coh['avg_msc_seizure_low'],
                Avg_MSC_Pre_Low=coh['avg_msc_pre_low'],
                Avg_MSC_Seizure_High=coh['avg_msc_seizure_high'],
                Avg_MSC_Pre_High=coh['avg_msc_pre_high'],
                Tau_ms_Seizure=coh['tau_ms_seizure'],
                Tau_ms_Pre=coh['tau_ms_pre'],
                Phase_R2_Seizure=coh['phase_r2_seizure'],
                Phase_R2_Pre=coh['phase_r2_pre'],
                Phase_Nbins_Seizure=coh['phase_nbins_seizure'],
                Phase_Nbins_Pre=coh['phase_nbins_pre'],
                Phase_Confidence_Seizure=coh['phase_confidence_seizure'],
                Phase_Confidence_Pre=coh['phase_confidence_pre'],
                Seizure_DT=row['SeizureStartDT'],
                Phase_Delay_Plot=phase_plot_path,
                Coherence_Plot=coh_plot_path,
                Correlation_Plot=corr_plot_path,))
            coherence_details.append({**coh,
                "Animal_ID": row['Animal_no'],
                "Seizure": int(row['Seizure'])})
            cfig = P.plot_coherence(
                coh['freqs_pre'], coh['msc_pre'], coh['freqs_seizure'], coh['msc_seizure'], sf,
                pre_seizure_secs=PRE_SEIZURE_SECS,
                title=f"Coherence | Animal {row['Animal_no']} Seiz {row['Seizure']}")
            cfig.savefig(coh_plot_path)
            coherence_plots.append(cfig)
            plt.close(cfig)

            pfig = P.plot_phase_delay_seizure(
                coh, band_low=COH_BAND_LOW_HZ,
                title=f"Phase Delay | Animal {row['Animal_no']} Seiz {row['Seizure']}")
            pfig.savefig(phase_plot_path)
            phase_delay_plots.append(pfig)
            plt.close(pfig)
        else:
            print(f"  ! Coherence skipped for Seizure {row['Seizure']}: pre-window out of bounds.")

    except Exception as e:
        print(f"  ERROR on Seizure {row['Seizure']} in {m.filename}: {e}")

correlation_results = pd.DataFrame(correlation_rows).sort_values('Animal_ID').reset_index(drop=True)
coherence_results = pd.DataFrame(coherence_rows).reset_index(drop=True)
print(f"\nAnalysis complete: {len(correlation_results)} correlation rows, "
      f"{len(coherence_results)} coherence rows.")

## 8. Correlation Plots

Rolling correlation per seizure, with zoomed raw segments.

In [ ]:
for fig in correlation_plots:
    display(fig)

## 9. Coherence Plots

Pre vs during-seizure MSC across frequency.

In [ ]:
for fig in coherence_plots:
    display(fig)

for fig in phase_delay_plots:
    display(fig)

## 10. Phase Delay Exploration

Per-animal MSC and phase-vs-frequency grids, plus a box plot of the delays. Reloads `plotting` first so module edits show up.

In [ ]:
import importlib
import plotting
P = importlib.reload(plotting)

In [ ]:
import os
from ipywidgets import interact, FloatSlider

os.makedirs(PHASE_DELAY_EXPLORATION_DIR, exist_ok=True)

msc_grid = P.plot_msc_by_animal(coherence_details, band_low=COH_BAND_LOW_HZ, which="seizure")
msc_grid.savefig(os.path.join(PHASE_DELAY_EXPLORATION_DIR, "MSC_By_Animal.png"))
display(msc_grid); plt.close(msc_grid)

grid = P.plot_phase_by_animal(coherence_details, band_low=COH_BAND_LOW_HZ,
                              which="seizure", color_by="msc", phase_ylim=(-3, 3))
grid.savefig(os.path.join(PHASE_DELAY_EXPLORATION_DIR, "Phase_By_Animal_MSCopacity.png"))
display(grid); plt.close(grid)

@interact(r2_min=FloatSlider(min=0.0, max=1, step=0.01, value=0.5,
                             continuous_update=True))
def _show(r2_min):
    fig = P.plot_delay_distribution(coherence_results, r2_min=r2_min, nbins_min=3)
    plt.show()

## 11. Results Tables

In [ ]:
display(correlation_results)

In [ ]:
display(coherence_results)

## 12. Per-Animal Correlation Summaries

Duration, time above threshold, strength and Racine severity over time.

In [ ]:
render_per_animal(
    correlation_results, CORRELATION_ANIMAL_DIR,
    lambda a: f"Animal_{a}_Summary.png",
    lambda g: P.plot_animal_summary(g, CORR_THRESHOLD, COLORMAP, RACINE_SCALE_MAX),
)

## 13. Per-Animal Coherence Summaries

Band-averaged coherence over time, and the seizure/pre-seizure MSC ratio.

In [ ]:
render_per_animal(
    coherence_results, COHERENCE_ANIMAL_DIR,
    lambda a: f"Animal_{a}_Coherence_Summary.png",
    lambda g: P.plot_coherence_animal_summary(g, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ,
                                              pre_seizure_secs=PRE_SEIZURE_SECS),
)

render_per_animal(
    coherence_results, COHERENCE_ANIMAL_DIR,
    lambda a: f"Animal_{a}_Coherence_Ratio.png",
    lambda g: P.plot_coherence_ratio_animal_summary(g, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ),
)

## 14. Per-Animal Phase Delay Summaries

Delay (ms) per seizure over time. Colour = phase-slope fit R², size = coherent bins. Saved to `Output/Phase_Delay/Per_Animal`.

In [ ]:
render_per_animal(
    coherence_results, PHASE_DELAY_ANIMAL_DIR,
    lambda a: f"Animal_{a}_Phase_Delay.png",
    lambda g: P.plot_phase_delay_animal_summary(g, COH_BAND_LOW_HZ,
                                               pre_seizure_secs=PRE_SEIZURE_SECS),
)

## 14b. Per-Animal During-Seizure Phase Delay (Context)

In [ ]:
render_per_animal(
    coherence_results, PHASE_DELAY_ANIMAL_DIR,
    lambda a: f"Animal_{a}_Phase_Delay_During_Context.png",
    lambda g: P.plot_phase_delay_during_animal_summary(g, COH_BAND_LOW_HZ,
                                                       racine_cmap=COLORMAP,
                                                       racine_max=RACINE_SCALE_MAX),
)

## 15. Global Correlation Summary (Offset to Day 0)

All animals on one experimental-day axis.

In [ ]:
norm_corr = offset_to_day_zero(correlation_results, day_zero=DAY_ZERO)
render_global(CORRELATION_GLOBAL_DIR, "Global_Correlation_Summary.png",
              P.plot_global_summary(norm_corr, RACINE_SCALE_MAX, DAY_ZERO))

## 16. Global Coherence Summary (Offset to Day 0)

Band-averaged coherence and the seizure/pre-seizure ratio, all animals on the Day 0 axis.

In [ ]:
norm_coh = offset_to_day_zero(coherence_results, day_zero=DAY_ZERO)
render_global(COHERENCE_GLOBAL_DIR, "Global_Coherence_Summary.png",
              P.plot_coherence_global_summary(norm_coh, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ,
                                              day_zero=DAY_ZERO))

In [ ]:
render_global(COHERENCE_GLOBAL_DIR, "Global_Coherence_Ratio.png",
              P.plot_coherence_ratio_global_summary(norm_coh, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ,
                                                    day_zero=DAY_ZERO))

## 17. Global Phase Delay Summary (Offset to Day 0)

One point per seizure (during-seizure delay). Colour = fit R², size = coherent bins, shape = animal.

In [ ]:
from ipywidgets import interact, FloatSlider

# decision: sweep frames get their own folder so they don't clutter the main output
GLOBAL_PHASE_DELAY_SWEEP_DIR = os.path.join(PHASE_DELAY_GLOBAL_DIR, "Global_Phase_Delay_Sweep")
os.makedirs(PHASE_DELAY_GLOBAL_DIR, exist_ok=True)
os.makedirs(GLOBAL_PHASE_DELAY_SWEEP_DIR, exist_ok=True)

# decision: lock axes to the full unfiltered range so every sweep frame shares one scale
_tau = norm_coh["Tau_ms_Seizure"].dropna()
_pad = 0.05 * (_tau.max() - _tau.min())
PHASE_DELAY_YLIM = (_tau.min() - _pad, _tau.max() + _pad)

_dt = norm_coh["Normalized_DT"]
_xpad = 0.02 * (_dt.max() - _dt.min())
PHASE_DELAY_XLIM = (_dt.min() - _xpad, _dt.max() + _xpad)

# explainer: confidence = R2 + bins + MSC folded into 0-1
@interact(conf_min=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.0, continuous_update=False))
def _show_global_phase_delay(conf_min):
    fig = P.plot_phase_delay_global_summary(norm_coh, COH_BAND_LOW_HZ, day_zero=DAY_ZERO,
                                            conf_min=conf_min, ylim=PHASE_DELAY_YLIM,
                                            xlim=PHASE_DELAY_XLIM)
    fig.savefig(os.path.join(PHASE_DELAY_GLOBAL_DIR, "Global_Phase_Delay.png"))
    plt.show()


# decision: one frame per 0.05 step, stop once nothing is left to plot
def save_global_phase_delay_sweep(step=0.05):
    conf = 0.0
    while (norm_coh["Phase_Confidence_Seizure"] >= conf).sum() > 0:
        fig = P.plot_phase_delay_global_summary(norm_coh, COH_BAND_LOW_HZ, day_zero=DAY_ZERO,
                                                conf_min=conf, ylim=PHASE_DELAY_YLIM,
                                                xlim=PHASE_DELAY_XLIM)
        fig.savefig(os.path.join(GLOBAL_PHASE_DELAY_SWEEP_DIR,
                                 f"Global_Phase_Delay_conf{conf:.2f}.png"))
        plt.close(fig)
        conf += step

save_global_phase_delay_sweep()

In [ ]:
# explainer: hover for stats, click a point to open that seizure's charts. same locked axes as the PNG sweep
phase_delay_html = os.path.join(PHASE_DELAY_GLOBAL_DIR, "Global_Phase_Delay_Interactive.html")
P.plot_phase_delay_global_interactive(
    norm_coh, phase_delay_html, band_low=COH_BAND_LOW_HZ, day_zero=DAY_ZERO,
    conf_min=0.0, conf_step=0.05, ylim=PHASE_DELAY_YLIM, xlim=PHASE_DELAY_XLIM)
print(f"Wrote {phase_delay_html}")

In [ ]:
# decision: public/ is what gets committed and deployed to Cloudflare Pages (Output/ is gitignored), PNGs sit next to the HTML so links work anywhere
import shutil

PUBLIC_DIR = "public"
PUBLIC_CHARTS = os.path.join(PUBLIC_DIR, "seizure_charts")
os.makedirs(PUBLIC_CHARTS, exist_ok=True)

# explainer: copy each seizure's PNGs in and repoint the path columns at the copies
pub_df = norm_coh.copy()
for col in ("Phase_Delay_Plot", "Coherence_Plot", "Correlation_Plot"):
    new_paths = []
    for p in pub_df[col]:
        if isinstance(p, str) and os.path.exists(p):
            dst = os.path.join(PUBLIC_CHARTS, os.path.basename(p))
            shutil.copy2(p, dst)
            new_paths.append(dst)
        else:
            new_paths.append("")
    pub_df[col] = new_paths

# decision: index.html so the bare site URL opens the chart
P.plot_phase_delay_global_interactive(
    pub_df, os.path.join(PUBLIC_DIR, "index.html"), band_low=COH_BAND_LOW_HZ,
    day_zero=DAY_ZERO, conf_min=0.0, conf_step=0.05,
    ylim=PHASE_DELAY_YLIM, xlim=PHASE_DELAY_XLIM)
print(f"Built {PUBLIC_DIR}/ -- git add public, commit, and push, then deploy it on Cloudflare Pages.")

## 17b. Global During-Seizure Phase Delay (Context)

In [ ]:
from ipywidgets import interact, FloatSlider

os.makedirs(PHASE_DELAY_GLOBAL_DIR, exist_ok=True)

# explainer: confidence = R2 + bins + MSC folded into 0-1
@interact(conf_min=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.0, continuous_update=False))
def _show_global_phase_delay_context(conf_min):
    fig = P.plot_phase_delay_during_global_summary(norm_coh, COH_BAND_LOW_HZ,
                                                   day_zero=DAY_ZERO,
                                                   racine_cmap=COLORMAP,
                                                   racine_max=RACINE_SCALE_MAX,
                                                   conf_min=conf_min)
    fig.savefig(os.path.join(PHASE_DELAY_GLOBAL_DIR, "Global_Phase_Delay_During_Context.png"))
    plt.show()